In [ ]:
# =========================
# 1. Install dependencies
# =========================
!pip install -q openai pandas tqdm

In [ ]:
# =========================
# 2. Imports
# =========================
import os
import json
import pandas as pd
import re
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive, files

In [ ]:
# =========================
# 3. Setup API key
# =========================
# Get your key from https://openrouter.ai
os.environ["OPENAI_API_KEY"] = "YOUR_OPENROUTER_API_KEY"

client = OpenAI(base_url="https://openrouter.ai/api/v1")

In [ ]:
# =========================
# 4. Helper: Extract JSON safely
# =========================
def extract_json(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(match.group()) if match else None
    except:
        return None

In [ ]:
# =========================
# 5. Query DeepSeek
# =========================
def query_deepseek_agent(system_prompt, user_message):
    try:
        response = client.chat.completions.create(
            model="deepseek/deepseek-chat-v3:free",  # or deepseek-r1:free
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0
        )

        text = response.choices[0].message.content
        return extract_json(text)

    except Exception as e:
        print(f"Error querying DeepSeek: {e}")
        return None

In [ ]:
# =========================
# 6. Define agents
# =========================

fact_system = """You are FactCheckerAgent.
Classify the text as:
- real
- fake(misinformation)
- unrelated

Return ONLY valid JSON:
{"label": "..."}"""

sentiment_system = """You are SentimentAgent.
Classify sentiment as:
- positive
- negative
- neutral

Return ONLY valid JSON:
{"sentiment": "..."}"""

combiner_system = """You are CombinerAgent.
Given text, label, and sentiment:
Fix any logical inconsistencies.

Return ONLY valid JSON:
{"label": "...", "sentiment": "..."}"""


In [ ]:
# =========================
# 7. Mount Google Drive
# =========================
drive.mount('/content/drive')

In [ ]:
# =========================
# 8. Load data
# =========================
data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.strip():
            try:
                reviews.append(json.loads(line))
            except:
                pass
        if len(reviews) >= 100:  # test limit
            break

print(f"Loaded {len(reviews)} reviews")

In [ ]:
# =========================
# 9. Process with 3 agents
# =========================
results = []

for row in tqdm(reviews, desc="DeepSeek 3-Agent Processing"):
    text = row.get("text", "")

    if not text.strip():
        label, sentiment = "unrelated", "neutral"
    else:
        # Agent 1: Fact
        fact_res = query_deepseek_agent(fact_system, f"Text: {text}")
        label = fact_res.get("label", "unrelated") if fact_res else "unrelated"

        # Agent 2: Sentiment
        sent_res = query_deepseek_agent(sentiment_system, f"Text: {text}")
        sentiment = sent_res.get("sentiment", "neutral") if sent_res else "neutral"

        # Agent 3: Combiner
        combiner_input = f"Text: {text}\nLabel: {label}\nSentiment: {sentiment}"
        final_res = query_deepseek_agent(combiner_system, combiner_input)

        if final_res:
            label = final_res.get("label", label)
            sentiment = final_res.get("sentiment", sentiment)

    results.append({
        **row,
        "label": label,
        "sentiment": sentiment
    })

In [ ]:
# =========================
# 10. Save results
# =========================
df = pd.DataFrame(results)

output_path = "/content/drive/MyDrive/Czech/deepseek_analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done! Saved to {output_path}")

# Download locally
files.download(output_path)